# 03 - 简单 RAG

本 notebook 直接连接到 notebook 02 的输出。

工作流程:

```text
用户查询 -> 检索器 -> 提示词 + LLM -> 答案
```

我们比较两种检索策略:
- 相似度搜索
- MMR(最大边际相关性)

**前置条件:**
- `01_setup_and_basics.ipynb`
- `02_embeddings_comparison.ipynb`
- `data/models/` 下存在本地 BGE 模型
- `data/vector_stores/huggingface_embeddings` 下存在 BGE 向量存储

**预计时长:** ~15 分钟

## 1. 加载本地模型和向量存储

复用 notebook 02 中创建的相同本地 BGE 模型和 FAISS 向量存储。

In [1]:
import sys
sys.path.append('../..')

from shared.config import PROJECT_ROOT, DEFAULT_K, HF_VECTOR_STORE_PATH, create_embeddings
from shared.utils import load_vector_store, print_section_header

print_section_header("加载向量存储")

MODEL_DIR = PROJECT_ROOT / 'data' / 'models'
existing_model_dirs = sorted(
    path.parent
    for path in MODEL_DIR.rglob('modules.json')
    if path.parent.name.startswith('bge-small-en-v1')
)

if not existing_model_dirs:
    raise FileNotFoundError(
        '在 data/models/ 下未找到本地 BGE 模型。请先运行 02_embeddings_comparison.ipynb。'
    )

model_dir = str(existing_model_dirs[0])
print(f"使用本地 BGE 模型: {model_dir}")

embeddings = create_embeddings(model_name=model_dir)

vectorstore = load_vector_store(
    HF_VECTOR_STORE_PATH,
    embeddings,
    verbose=True,
)

if vectorstore is None:
    raise FileNotFoundError(
        f'无法从 {HF_VECTOR_STORE_PATH} 加载向量存储。请先运行 02_embeddings_comparison.ipynb。'
    )

print("\n向量存储已加载并准备就绪")
print(f"向量存储路径: {HF_VECTOR_STORE_PATH}")
print(f"每次查询使用 k={DEFAULT_K} 个文档")


加载向量存储

使用本地 BGE 模型: d:\ZKS_Data\AI\langchain-rag-tutorial\notebooks\fundamentals\..\..\data\models\BAAI\bge-small-en-v1___5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded vector store from d:\ZKS_Data\AI\langchain-rag-tutorial\notebooks\fundamentals\..\..\data\vector_stores\huggingface_embeddings

向量存储已加载并准备就绪
向量存储路径: d:\ZKS_Data\AI\langchain-rag-tutorial\notebooks\fundamentals\..\..\data\vector_stores\huggingface_embeddings
每次查询使用 k=4 个文档


## 2. 构建检索器

在向量存储之上创建相似度和 MMR 检索器。

In [2]:
from shared.config import DEFAULT_MMR_FETCH_K, DEFAULT_MMR_LAMBDA

print_section_header("创建检索器")

similarity_retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': DEFAULT_K},
)
print(f"相似度检索器已创建 (k={DEFAULT_K})")

mmr_retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={
        'k': DEFAULT_K,
        'fetch_k': DEFAULT_MMR_FETCH_K,
        'lambda_mult': DEFAULT_MMR_LAMBDA,
    },
)
print(
    f"MMR 检索器已创建 (k={DEFAULT_K}, fetch_k={DEFAULT_MMR_FETCH_K}, lambda={DEFAULT_MMR_LAMBDA})"
)



创建检索器

相似度检索器已创建 (k=4)
MMR 检索器已创建 (k=4, fetch_k=20, lambda=0.5)


## 3. 比较检索结果

对两个检索器运行相同的查询。

In [3]:
from shared.utils import print_results

query = 'What are the steps to build a RAG agent?'

print_section_header('检索比较')
print(f"查询: {query}\n")

print('=== 相似度搜索 ===')
similarity_docs = similarity_retriever.invoke(query)
print_results(similarity_docs, '检索到的文档', max_docs=3, preview_length=150)

print('\n' + '=' * 80)
print('=== MMR 搜索 ===')
mmr_docs = mmr_retriever.invoke(query)
print_results(mmr_docs, '检索到的文档', max_docs=3, preview_length=150)



检索比较

查询: What are the steps to build a RAG agent?

=== 相似度搜索 ===

检索到的文档
--------------------------------------------------------------------------------

1. Source: https://python.langchain.com/docs/use_cases/question_answering/
   Type: local_text_download
   Date: 2026-05-09
   Content: We will demonstrate:
A RAG
agent
that executes searches with a simple tool. This is a good general-purpose implementation.
A two-step RAG
chain
that u...

2. Source: https://python.langchain.com/docs/use_cases/chatbots/
   Type: local_text_download
   Date: 2026-05-09
   Content: We will demonstrate:
A RAG
agent
that executes searches with a simple tool. This is a good general-purpose implementation.
A two-step RAG
chain
that u...

3. Source: https://python.langchain.com/docs/use_cases/question_answering/
   Type: local_text_download
   Date: 2026-05-09
   Content: prompt injection
.
​
Next steps
Now that we’ve implemented a simple RAG application via
create_agent
, we can easily incorporate new fe

## 4. 初始化 LLM

使用配置的聊天模型进行答案生成。

In [4]:
from shared.config import create_chat_model, DEFAULT_MODEL, DEFAULT_TEMPERATURE

print_section_header('初始化 LLM')

llm = create_chat_model(
    model=DEFAULT_MODEL,
    temperature=DEFAULT_TEMPERATURE,
)

print('LLM 已初始化')
print(f'模型: {DEFAULT_MODEL}')
print(f'温度: {DEFAULT_TEMPERATURE}')


初始化 LLM



langchain-openai injected a custom httpx transport to apply `http_socket_options`, which disables httpx's proxy auto-detection (system proxy configuration detected). Set `LANGCHAIN_OPENAI_TCP_KEEPALIVE=0` or pass `http_socket_options=()` to restore default proxy behavior, or supply `openai_proxy` / your own `http_client` / `http_async_client` to take full control.


LLM 已初始化
模型: deepseek-v4-flash
温度: 0.0


## 5. 构建 RAG 链

为相似度搜索和 MMR 各创建一个链。

In [5]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from shared.prompts import RAG_PROMPT_TEMPLATE
from shared.utils import format_docs

print_section_header('构建 RAG 链')

similarity_chain = (
    {'context': similarity_retriever | format_docs, 'input': RunnablePassthrough()}
    | RAG_PROMPT_TEMPLATE
    | llm
    | StrOutputParser()
)
print('相似度 RAG 链已创建')

mmr_chain = (
    {'context': mmr_retriever | format_docs, 'input': RunnablePassthrough()}
    | RAG_PROMPT_TEMPLATE
    | llm
    | StrOutputParser()
)
print('MMR RAG 链已创建')


构建 RAG 链

相似度 RAG 链已创建
MMR RAG 链已创建


## 6. 比较答案

使用两种检索策略生成答案。

In [6]:
import time

query = 'How to build a RAG agent with LangChain?'

print_section_header('RAG 链比较')
print(f"查询: {query}\n")

print('=' * 80)
print('相似度 RAG')
print('=' * 80)
start = time.time()
similarity_answer = similarity_chain.invoke(query)
similarity_elapsed = time.time() - start
print(similarity_answer)
print(f"\n耗时: {similarity_elapsed:.2f}s")

print('\n' + '=' * 80)
print('MMR RAG')
print('=' * 80)
start = time.time()
mmr_answer = mmr_chain.invoke(query)
mmr_elapsed = time.time() - start
print(mmr_answer)
print(f"\n耗时: {mmr_elapsed:.2f}s")


RAG 链比较

查询: How to build a RAG agent with LangChain?

相似度 RAG
Based on the provided context, building a RAG (Retrieval-Augmented Generation) agent with LangChain involves two main phases: **Indexing** and **Retrieval & Generation**. The documentation outlines these components at a high level:

### 1. Indexing
- **Loading documents** – Ingest your data source (e.g., PDFs, web pages).
- **Splitting documents** – Break the loaded text into smaller, manageable chunks.
- **Storing documents** – Index the chunks (e.g., in a vector store) to enable fast retrieval.

### 2. Retrieval and Generation
- **RAG agents** – Use a LangChain agent (e.g., `create_agent`) that combines a retriever with an LLM to answer queries based on the indexed knowledge.
- **RAG chains** – Alternatively, you can build a simpler retrieval chain that passes retrieved context to the LLM.
- **Security considerations** – The documentation also mentions **indirect prompt injection** as a security aspect to be aware of.

T

## 7. 批量测试

通过相同的管道运行几个额外的问题。

In [7]:
queries = [
    'What is a retriever?',
    'How do embeddings work?',
    'What are the key components of a RAG system?',
]

print_section_header('批量测试')

for i, query in enumerate(queries, start=1):
    print(f"\n[{i}] 查询: {query}")
    answer = similarity_chain.invoke(query)
    print(answer[:500])
    if len(answer) > 500:
        print('...')


批量测试


[1] 查询: What is a retriever?
Based on the context provided, a **retriever** is a component used in RAG applications to retrieve relevant splits (chunks of data) from storage given a user input. Specifically, the context states:

> "Retrieve: Given a user input, relevant splits are retrieved from storage using a **Retriever**."

[2] 查询: How do embeddings work?
Based on the provided context, embeddings are **numerical representations of text** generated by a text embedding model. They are used to convert text (like document splits) into a format that can be stored in a vector store and later used for similarity search.

The context describes the process as: "our approach is to embed the contents of each document split and insert these embeddings into a vector store. Given an input query, we can then use vector search to retrieve relevant documents."

I
...

[3] 查询: What are the key components of a RAG system?
Based on the context, the key components of a RAG system are organized 

## 总结

在本 notebook 中,我们:
- 加载了 notebook 02 中创建的本地 BGE 模型
- 加载了 notebook 02 中创建的 BGE FAISS 向量存储
- 构建了相似度和 MMR 检索器
- 创建了简单的 RAG 链
- 比较了检索和答案生成结果

下一步:
- 继续到 [04_rag_with_memory.ipynb](../advanced_architectures/04_rag_with_memory.ipynb)